<a href="https://colab.research.google.com/github/surajjeoor/masai_assignments/blob/main/Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
PRICES = {  # $ per 1M tokens
    "gpt-5.4-mini": {"input": 0.75, "cached": 0.075, "output": 4.50},
    "gpt-5.4": {"input": 2.50, "cached": 0.25, "output": 15.00},
}

REQUESTS = [
    ("gpt-5.4-mini", {"prompt_tokens": 2200, "cached_tokens": 0, "completion_tokens": 40}),
    ("gpt-5.4-mini", {"prompt_tokens": 2201, "cached_tokens": 2048, "completion_tokens": 35}),
    ("gpt-5.4", {"prompt_tokens": 2210, "cached_tokens": 2048, "completion_tokens": 50}),
]

def cost_of(usage, model):
    p = PRICES[model]
    total_price = 0.0
    # TODO: work out the fresh input tokens
    input_price=p["input"]
    cached_price=p["cached"]
    output_price=p["output"]
    if usage["cached_tokens"] > 0:
      fresh_tokens = usage["prompt_tokens"] - usage["cached_tokens"]
      total_price =(fresh_tokens * input_price) + (usage["cached_tokens"] * cached_price) + (usage["completion_tokens"] * output_price)
      return total_price/1000000
    else:
      total_price=(usage["prompt_tokens"] * input_price) + (usage["completion_tokens"] * output_price)
      return total_price/1000000
    # TODO: price fresh, cached and completion tokens and divide by 1,000,000
    return 0.0

for i, (model, usage) in enumerate(REQUESTS, start=1):
    print(f"Request {i} ({model}): ${cost_of(usage, model):.6f}")

monthly = cost_of(REQUESTS[1][1], "gpt-5.4-mini") * 100_000 * 30
print(f"Monthly bill at request 2's rate: ${monthly:,.2f}")

Request 1 (gpt-5.4-mini): $0.001830
Request 2 (gpt-5.4-mini): $0.000426
Request 3 (gpt-5.4): $0.001667
Monthly bill at request 2's rate: $1,277.55


In [4]:
THRESHOLD = 0.85
cache = []  # list of (question, answer); Redis in production

SCORES = {
    ("Is the delivery free?", "Can I return my order?"): 0.20,
    ("Can I get free delivery?", "Can I return my order?"): 0.18,
    ("Can I get free delivery?", "Is the delivery free?"): 0.91,
    ("Can I cancel my order?", "Can I return my order?"): 0.74,
    ("Can I cancel my order?", "Is the delivery free?"): 0.15,
    ("When will I get my refund?", "Can I return my order?"): 0.31,
    ("When will I get my refund?", "Is the delivery free?"): 0.12,
    ("When will I get my refund?", "Can I cancel my order?"): 0.27,
    ("How long do refunds take?", "When will I get my refund?"): 0.88,
}
def similarity(a, b):
    return SCORES.get((a, b), SCORES.get((b, a), 0.0))

def ask(question):
    return f"LLM answer to: {question}", 0.0018  # stands in for the LLM call

def cached_ask(question):
    # TODO: find the best-scoring cached question and its answer
    best_score, best_answer = 0.0, ""
    for cached_question, cached_answer in cache:
      score = similarity(question, cached_question)
      if score > best_score:
        best_score = score
        best_answer = cached_answer

    # TODO: on a hit, return the stored answer, 0.0 and "hit"
    if best_score > THRESHOLD:
      return best_answer, 0.0, "hit"
    # TODO: on a miss, call ask(), store the pair, and return answer, cost, "miss"
    answer, cost = ask(question)
    cache.append((question, answer))
    return answer, cost, "miss"

QUESTIONS = [
    "Can I return my order?",
    "Is the delivery free?",
    "Can I get free delivery?",
    "Can I cancel my order?",
    "When will I get my refund?",
    "How long do refunds take?",
]

total, hits = 0.0, 0
for q in QUESTIONS:
    answer, cost, status = cached_ask(q)
    total += cost
    hits += status == "hit"
    print(f"{status:<5} {cost:.4f}  {answer}")
print(f"Hits: {hits} of {len(QUESTIONS)}")
print(f"Total cost: ${total:.4f}")

miss  0.0018  LLM answer to: Can I return my order?
miss  0.0018  LLM answer to: Is the delivery free?
hit   0.0000  LLM answer to: Is the delivery free?
miss  0.0018  LLM answer to: Can I cancel my order?
miss  0.0018  LLM answer to: When will I get my refund?
hit   0.0000  LLM answer to: When will I get my refund?
Hits: 2 of 6
Total cost: $0.0072


In [7]:
import asyncio
import time

LIMIT = 4
TOPIC_WORDS = [("refund", "refund"), ("deliver", "delivery"), ("app", "app"),
               ("support", "support"), ("order", "order")]
NEGATIVE_WORDS = ["not", "pathetic", "late", "crash"]
stats = {"in_flight": 0, "peak": 0}

REVIEWS = [
    "Delivered in two days and the packaging was excellent",
    "Refund still not received after 10 days, pathetic service",
    "The app keeps crashing at checkout",
    "Great phone, works well and looks premium",
    "Order arrived late and the box was torn",
    "Support team solved my issue quickly",
    "Delivery boy was rude and very late",
    "Got my refund within three days, nice",
    "The shirt fabric is soft and fits well",
    "The app is smooth and easy to use",
]

async def tag_review_async(review):
    stats["in_flight"] += 1
    stats["peak"] = max(stats["peak"], stats["in_flight"])
    await asyncio.sleep(0.1)  # stands in for the API call
    stats["in_flight"] -= 1
    text = review.lower()
    sentiment = "positive"
    for word in NEGATIVE_WORDS:
        if word in text:
            sentiment = "negative"
    topic = "product"
    for word, label in TOPIC_WORDS:
        if word in text:
            topic = label
            break
    return f"{sentiment}, {topic}"

async def tag_all(reviews, limit):
    # Create a semaphore that allows `limit` requests at once
    semaphore = asyncio.Semaphore(limit)

    async def sem_task(review):
        async with semaphore:
            return await tag_review_async(review)

    # Run each tag_review_async call inside the semaphore
    tasks = [sem_task(review) for review in reviews]
    results = await asyncio.gather(*tasks)
    return results

async def main():
    # Reset stats before run
    stats["in_flight"] = 0
    stats["peak"] = 0
    start = time.time()
    tags = await tag_all(REVIEWS, LIMIT)
    elapsed = time.time() - start
    for i, tag in enumerate(tags, start=1):
        print(f"{i}. {tag}")
    print(f"Peak in flight: {stats['peak']}")
    print(f"Elapsed: about {elapsed:.1f} seconds")

# In Jupyter/Colab, run the async function directly with top-level await
await main()

/usr/lib/python3.13/asyncio/locks.py:13: RuntimeWarning: coroutine 'main' was never awaited
  async def __aenter__(self):


1. positive, delivery
2. negative, refund
3. negative, app
4. positive, product
5. negative, order
6. positive, support
7. negative, delivery
8. positive, refund
9. positive, product
10. positive, app
Peak in flight: 4
Elapsed: about 0.3 seconds
